In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def drop_cols(df, cols=[]):
    #Cleans redundant columns (if there's no values for any datasets in the column, remove that specific column)
    print(f"Original columns: {len(df.columns)}")
    #Further refines the column field by removing repeat columns 
    filtered_df = df.drop(columns=cols)
    print(f"Dropped columns: {len(cols)}")
    print(f"Remaining columns: {len(filtered_df.columns)}")
    return filtered_df


## Filter to all spectroscopy done on gases 

In [ ]:
#Script to trim down the 1.3GB dataset by filtering for gas only states
t = pd.read_csv(r"/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/NIST_IR_Spectroscopy.csv",low_memory=False) #Used relative path based on my local machine. Change as needed

gas_df= t[t['state'].str.contains('gas',case=False,na=False)]

print(f"Original rows:{len(t)}")
print(f"Gas rows: {len(gas_df)}")

#Cleans redundant columns (if there's no values for any datasets in the column, remove that specific column)
gas_df_columnCleaned = gas_df.dropna(axis=1, how='all')
redundant_axis_dropped = ['maxx', 'minx', 'maxy', 'miny']
gas_df_columnCleaned = drop_cols(gas_df_columnCleaned, redundant_axis_dropped)


## Feature selection

### what is being dropped
* mostly null cols
* columns with low variance / unique to each row(id)
* columns that can be derived from smiles (melting point, name of compound, molecular formula, etc)
* cols that have high correlation with another

In [ ]:
post_feature_selection_rows = ["filename","smiles", "xunits", "yunits", "xfactor",  "yfactor", "x_coords", "y_coords"]
selected_df = gas_df_columnCleaned[post_feature_selection_rows]


## Standardize x,y coordinate data

### convert all x units to 1/cm

In [ ]:
print("number of rows before filtering out x units, ",selected_df.count())
filtered_x_unit_df=selected_df[selected_df["xunits"].isin(["1/CM", "cm-1"])]
print("number of rows after, ",filtered_x_unit_df.count())

In [ ]:
import ast
filtered_x_unit_df["x_coords"] = filtered_x_unit_df["x_coords"].apply(ast.literal_eval)
filtered_x_unit_df["x_coords"] = filtered_x_unit_df.apply(
    lambda row: [coord * float(row["xfactor"]) for coord in row["x_coords"]],
    axis=1
)

In [ ]:
#drop x factor
filtered_x_unit_df = drop_cols(filtered_x_unit_df, ["xfactor"])

### normalize x coordinate domain

In [ ]:
filtered_x_unit_df["y_coords"] = filtered_x_unit_df["y_coords"].apply(ast.literal_eval)
def trim_xy_pairs(row, start_val, end_val):
    x_coords = row["x_coords"]
    y_coords = row["y_coords"]
    
    pairs = list(zip(x_coords, y_coords))
    
    # Trim from the front: remove pairs until x >= start_val
    start = 0
    while start < len(pairs) and pairs[start][0] < start_val:
        start += 1
    
    # Trim from the back: remove pairs until x <= end_val
    end = len(pairs) - 1
    while end >= start and pairs[end][0] > end_val:
        end -= 1
    
    trimmed = pairs[start:end + 1]
    
    if trimmed:
        row["x_coords"], row["y_coords"] = zip(*trimmed)
        row["x_coords"] = list(row["x_coords"])
        row["y_coords"] = list(row["y_coords"])
    else:
        row["x_coords"] = []
        row["y_coords"] = []
    
    return row

normalized_x_df = filtered_x_unit_df.apply(trim_xy_pairs,start_val=400, end_val=4000, axis=1)


print(f"Rows after filtering: {len(normalized_x_df)}")
print("Filtered DataFrame head:")
print(normalized_x_df.head())

### normalize all y units (transmitance, etc..) to absorbance

In [ ]:
normalized_x_df=pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/notebooks/normalized_x_df.csv")

##### y factor already applied during jdx data extraction. remove it 

In [ ]:
normalized_x_df = normalized_x_df.drop("yfactor", axis=1)

In [ ]:
# Make a copy to avoid modifying the original DataFrame directly and to ensure operations are on a fresh copy
filtered_y_unit_df = normalized_x_df.copy()


# Ensure y_coords are lists, not strings (necessary if loaded from CSV)
filtered_y_unit_df['y_coords'] = filtered_y_unit_df['y_coords'].apply(ast.literal_eval)

##### filter out y units of `micromol/mol)-1m-1 (base 10)`
* if we need the 1000 extra data points we could go back to jdx and properly converty it to absorption using jdx metadata

In [ ]:
print("number of rows before filtering out y units, ",filtered_y_unit_df.count())
filtered_y_unit_df=filtered_y_unit_df[filtered_y_unit_df["yunits"].isin(["ABSORBANCE", "TRANSMITTANCE"])]
print("number of rows after, ",filtered_y_unit_df.count())

##### convert transmitance percentage to abs absorbance val

In [ ]:
import ast
import numpy as np

def apply_log_conversion(row):
    # Assumes yfactor is already applied to y_coords values if needed
    y_raw_list = row['y_coords']
    # Add a small epsilon to avoid log(0) which can happen with raw data near zero
    epsilon = 1e-9
    return [float(-np.log10(max(epsilon, val))) for val in y_raw_list]

# 1. Handle 'TRANSMITTANCE' conversion
transmittance_mask = filtered_y_unit_df['yunits'] == 'TRANSMITTANCE'
filtered_y_unit_df.loc[transmittance_mask, 'y_coords'] = (
    filtered_y_unit_df[transmittance_mask].apply(apply_log_conversion, axis=1)
)

### trim y values range

In [ ]:
def trim_xy_pairs(row, start_val, end_val):
    x_coords = row["x_coords"]
    y_coords = row["y_coords"]
    
    pairs = list(zip(x_coords, y_coords))
    
    # Trim from the front: remove pairs until y >= start_val
    start = 0
    while start < len(pairs) and pairs[start][1] < start_val:
        start += 1
    
    # Trim from the back: remove pairs until y <= end_val
    end = len(pairs) - 1
    while end >= start and pairs[end][1] > end_val:
        end -= 1
    
    trimmed = pairs[start:end + 1]
    
    if trimmed:
        row["x_coords"], row["y_coords"] = zip(*trimmed)
        row["x_coords"] = list(row["x_coords"])
        row["y_coords"] = list(row["y_coords"])
    else:
        row["x_coords"] = []
        row["y_coords"] = []
    
    return row

y_lower_bound = 0.0
y_upper_bound = 1.05

filtered_y_range_df = filtered_y_unit_df.apply(
    trim_xy_pairs, 
    start_val=y_lower_bound, 
    end_val=y_upper_bound, 
    axis=1
)


In [ ]:
filtered_y_range_df.drop(filtered_y_range_df.columns[filtered_y_range_df.columns.str.contains('unnamed', case=False)], axis=1, inplace=True)


In [ ]:
filtered_y_range_df.to_csv("SMILES_IR_Spectroscopy.csv", index=False)

### could also perform normalizaiton on absorbance values, but that depends on model
* if using cnn, L2 normalization is recommended
* otherwise, min max normalization